### Notebook Execution Order

- To ensure the project runs correctly, you have to make sure that the notebook `23127130_23127168_23127231_23127238_23127269.ipynb` **MUST** be run first, because it performs the dataset merging and preparation steps required for the other notebooks.

- After that, the remaining notebooks can be run, but to follow the order of the Objective, you should run like this order:
    1. `23127130_23127168_23127231_23127238_23127269.ipynb`
    2. `23127130_23127238.ipynb`
    3. `23127168.ipynb`
    4. `23127231_23127269.ipynb`

### YouTube Presentation

- Presentation video link: [YouTube link](YOUR_YOUTUBE_LINK_HERE)


## I. Introduction

This notebook is used to merging the data getting from the [WDI website](https://databank.worldbank.org/source/world-development-indicators) as well as introduce, summarize, and preprocess the dataset we choose for other notebooks that perform analysis and evaluation.


## II. Used libraries

This project uses several common Python libraries for data loading, processing, and visualization.

### `pandas`
`pandas` is the main library used to work with tabular data.

In our project, it is used to:
- read `.csv` files into DataFrames
- clean missing values such as `..`
- reshape data from wide format to long format using `melt()`
- merge datasets together
- group, sort, and summarize data
- prepare data for plotting

---

### `numpy`
`numpy` is used for numerical operations.

In this project, it can help with:
- handling numeric arrays efficiently
- supporting missing values and mathematical operations
- creating derived calculations if needed

Even if it is not heavily used in every notebook cell, it is useful when working with numerical analysis and statistics.

---

### `seaborn`
`seaborn` is a statistical data visualization library built on top of `matplotlib`.

In our project, it is used to:
- create clearer and more attractive charts
- visualize relationships between variables
- generate plots such as heatmaps, scatter plots, regression plots, and line plots

---

### `matplotlib.pyplot`
`matplotlib.pyplot` is the core plotting library.

In our project, it is used to:
- create and customize figures
- control chart size, titles, labels, and layout
- display multiple subplots in one figure

---

### `matplotlib.lines.Line2D`
`Line2D` is a class from `matplotlib.lines` that helps create custom line elements.

In our project, it is useful for:
- drawing custom legend entries for dumbbell charts
- representing markers and connecting lines clearly
- improving the readability of comparisons between two values
- customizing the appearance of chart elements beyond default settings

This is especially helpful when we want to make the dumbbell chart more informative and visually clean.

---

### `os`
`os` is used to work with file paths and folders in the operating system.

In our project, `os` is especially useful when:
- datasets are too large and must be downloaded in multiple parts
- several `.csv` files need to be stored in the same folder
- we want to build file paths dynamically instead of typing each file name manually
- we need to scan a folder and combine many split files into one dataset

This makes the project more flexible and easier to run when the raw data is stored in multiple parts.

---

### Note:

Some of the libraries may not import in this notebook since the main purpose of this notebook is introducing and preparing the dataset for the analysis stage.


## III. Importing libraries and building support functions

Import neccessary libraries

In [11]:
import pandas as pd
import numpy as np
import os

Merging the dataset into one

In [12]:
ID_COLUMNS = ["Country Name", "Country Code", "Series Name", "Series Code"]
MERGE_KEYS = ["Country Name", "Year"]


def _read_dataset(dataset_path: str) -> pd.DataFrame:
    for encoding in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(dataset_path, encoding=encoding)
        except UnicodeDecodeError:
            pass
    return pd.read_csv(dataset_path, encoding="latin1")


def clean_and_sort_dataset(dataset_path: str) -> pd.DataFrame:
    df = _read_dataset(dataset_path)
    df = df.iloc[:-5].replace("..", pd.NA)

    year_cols = [col for col in df.columns if "YR" in col]

    df_long = df.melt(
        id_vars=ID_COLUMNS,
        value_vars=year_cols,
        var_name="Year",
        value_name="Value",
    )

    df_long["Year"] = df_long["Year"].str.extract(r"(\d{4})").astype(int)
    df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
    df_long = df_long.sort_values("Year")

    df_final = (
        df_long.pivot_table(
            index=MERGE_KEYS,
            columns="Series Name",
            values="Value",
            aggfunc="mean",
        )
        .reset_index()
        .sort_values(MERGE_KEYS)
        .reset_index(drop=True)
    )

    df_final.columns.name = None
    return df_final


FOLDER_PATH = "../dataset"

def get_dataset_paths(folder_path: str) -> list:
    dataset_paths = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            dataset_paths.append(os.path.join(folder_path, filename))
    return dataset_paths


def merge_all_datasets(folder_path: str) -> pd.DataFrame:
    dataset_paths = get_dataset_paths(folder_path)
    if not dataset_paths:
        raise FileNotFoundError(f"No CSV files found in {folder_path}.")

    merged_df = clean_and_sort_dataset(dataset_paths[0])

    for dataset_path in dataset_paths[1:]:
        print(f"Processing {dataset_path}...")
        next_df = clean_and_sort_dataset(dataset_path)

        merged_df = merged_df.merge(
            next_df,
            on=MERGE_KEYS,
            how="outer",
            suffixes=("", "_new"),
        )

        overlapping_columns = [
            col for col in merged_df.columns
            if col.endswith("_new")
        ]

        for new_col in overlapping_columns:
            original_col = new_col.replace("_new", "")
            merged_df[original_col] = merged_df[original_col].combine_first(merged_df[new_col])
            merged_df = merged_df.drop(columns=new_col)

    return merged_df.sort_values(MERGE_KEYS).reset_index(drop=True)

print("Merging datasets...")
data = merge_all_datasets(FOLDER_PATH)
print("Datasets merged successfully.")


Merging datasets...
Processing ../dataset\902bcd0c-744b-4241-95a7-e91ee25d4ae4_Series - Metadata.csv...
Datasets merged successfully.


## IV. Overview about the dataset

- This dataset provides a multidimensional view of socio-economic conditions in Southeast Asia during the period **2000-2020**. It covers a group of Southeast Asian countries and brings together indicators from several major development domains, including **education, health, economic policy and debt**. By organizing these indicators in a consistent country-year structure, the dataset enables both cross-country comparison and temporal analysis across two decades of development change.

- The selected time range from **2000 to 2020** is especially meaningful because it captures a period of substantial transformation in Southeast Asia. During these years, many countries in the region experienced rapid economic growth, structural change, expansion in public services, and major policy efforts aimed at improving human development outcomes. As a result, the dataset is suitable for examining not only the level of development in each country, but also the pace and direction of change over time. 

In [13]:
SEA_COUNTRIES = [
    "Brunei Darussalam",
    "Cambodia",
    "Indonesia",
    "Lao PDR",
    "Malaysia",
    "Myanmar",
    "Philippines",
    "Singapore",
    "Thailand",
    "Timor-Leste",
    "Viet Nam",
]


# Ad more here if needed
CORE_VARS = [
    # Education indicators: Question 1
    "Government expenditure on education, total (% of GDP)",
    "Government expenditure on education, total (% of government expenditure)",
    "Pupil-teacher ratio, primary",
    "Pupil-teacher ratio, secondary",
    "Trained teachers in primary education (% of total teachers)",
    "Trained teachers in secondary education (% of total teachers)",
    "Children out of school (% of primary school age)",
    "School enrollment, primary (% net)",
]

# General dataset statistics without showing the variable list
dataset_statistics = pd.DataFrame(
    {
        "Statistic": [
            "Number of rows",
            "Number of columns",
            "Number of countries",
            "Time period",
            "Min year",
            "Max year",
            "Total missing values",
            "Duplicate country-year rows",
        ],
        "Value": [
            len(data),
            data.shape[1],
            data["Country Name"].nunique(),
            f"{data['Year'].min()}-{data['Year'].max()}",
            data["Year"].min(),
            data["Year"].max(),
            int(data.isna().sum().sum()),
            int(data.duplicated(subset=["Country Name", "Year"]).sum()),
        ],
    }
)

display(dataset_statistics)


,Statistic,Value
0,Number of rows,231
1,Number of columns,62
2,Number of countries,11
3,Time period,2000-2020
4,Min year,2000
5,Max year,2020
6,Total missing values,4046
7,Duplicate country-year rows,0


In [14]:
# Print all the columns in the dataset
print("Columns in the dataset:")
for col in data.columns:
    print(f"- {col}")

Columns in the dataset:
- Country Name
- Year
- Adolescents out of school (% of lower secondary school age)
- Children out of school (% of primary school age)
- Current account balance (% of GDP)
- Current health expenditure (% of GDP)
- Current health expenditure per capita, PPP (current international $)
- Debt service (PPG and IMF only, % of exports of goods, services and primary income)
- Domestic general government health expenditure (% of general government expenditure)
- Educational attainment, Doctoral or equivalent, population 25+, female (%) (cumulative)
- Educational attainment, Doctoral or equivalent, population 25+, male (%) (cumulative)
- Educational attainment, Doctoral or equivalent, population 25+, total (%) (cumulative)
- Educational attainment, at least Bachelor's or equivalent, population 25+, female (%) (cumulative)
- Educational attainment, at least Bachelor's or equivalent, population 25+, male (%) (cumulative)
- Educational attainment, at least Bachelor's or equi

In [15]:
# Print statistics for each column one by one
for col in data.columns:
    print(f"\n{'='*80}")
    print(f"Column: {col}")
    print(f"{'='*80}")
    print(f"Data type: {data[col].dtype}")
    print(f"Missing values: {data[col].isna().sum()}")
    print(f"Unique values: {data[col].nunique()}")

    if pd.api.types.is_numeric_dtype(data[col]):
        print(f"Mean: {data[col].mean()}")
        print(f"Std: {data[col].std()}")
        print(f"Min: {data[col].min()}")
        print(f"25%: {data[col].quantile(0.25)}")
        print(f"Median: {data[col].median()}")
        print(f"75%: {data[col].quantile(0.75)}")
        print(f"Max: {data[col].max()}")
    else:
        print("Top values:")
        print(data[col].value_counts(dropna=False).head(10))



Column: Country Name
Data type: object
Missing values: 0
Unique values: 11
Top values:
Country Name
Brunei Darussalam    21
Cambodia             21
Indonesia            21
Lao PDR              21
Malaysia             21
Myanmar              21
Philippines          21
Singapore            21
Thailand             21
Timor-Leste          21
Name: count, dtype: int64

Column: Year
Data type: int64
Missing values: 0
Unique values: 21
Mean: 2010.0
Std: 6.068450128041075
Min: 2000
25%: 2005.0
Median: 2010.0
75%: 2015.0
Max: 2020

Column: Adolescents out of school (% of lower secondary school age)
Data type: float64
Missing values: 81
Unique values: 150
Mean: 19.143566189882584
Std: 16.49151226668506
Min: 0.0646251863563981
25%: 9.916988138615144
Median: 14.183316800426201
75%: 24.1658674862409
Max: 84.9968490600586

Column: Children out of school (% of primary school age)
Data type: float64
Missing values: 75
Unique values: 153
Mean: 6.211913161706693
Std: 5.360247737329246
Min: 0.0
25%: 1.9

# V. Preprocessing data

Data preprocessing is an important step in the analysis workflow because raw data is often incomplete, inconsistent, or not yet structured in a form suitable for analysis and visualization. In this project, preprocessing helps ensure that the dataset is clean, readable, and ready for statistical exploration.

## Handling missing values

- For each country, missing numeric values are then filled over time using linear interpolation, followed by forward fill and backward fill to handle remaining gaps at the beginning or end of the series. 
- This makes the dataset more complete and reliable for later analysis and visualization.

In [16]:
data = data.sort_values(MERGE_KEYS).reset_index(drop=True)

numeric_cols = [
    col for col in data.columns
    if col not in MERGE_KEYS and pd.api.types.is_numeric_dtype(data[col])
]

# Fill missing indicator values within each country across time.
data[numeric_cols] = data.groupby("Country Name")[numeric_cols].transform(
    lambda values: values.interpolate().ffill().bfill()
)


In [17]:
data_imputed = data.copy()

n_years = data_imputed["Year"].nunique()
remaining_missing = data_imputed[numeric_cols].isna().sum()

# Track which values were imputed
imputed_flag = pd.DataFrame(False, index=data_imputed.index, columns=numeric_cols)

# 1. Deterministic fill: Trade = Exports + Imports
trade_col = "Trade (% of GDP)"
exports_col = "Exports of goods and services (% of GDP)"
imports_col = "Imports of goods and services (% of GDP)"

if {trade_col, exports_col, imports_col}.issubset(data_imputed.columns):
    mask = (
        data_imputed[trade_col].isna()
        & data_imputed[exports_col].notna()
        & data_imputed[imports_col].notna()
    )
    data_imputed.loc[mask, trade_col] = (
        data_imputed.loc[mask, exports_col] + data_imputed.loc[mask, imports_col]
    )
    imputed_flag.loc[mask, trade_col] = True

# 2. Safe fill: fill columns missing for exactly one country across all years (21 values)
one_country_missing_cols = remaining_missing[remaining_missing == n_years].index.tolist()

for col in one_country_missing_cols:
    year_median = data_imputed.groupby("Year")[col].transform("median")
    mask = data_imputed[col].isna() & year_median.notna()
    data_imputed.loc[mask, col] = year_median[mask]
    imputed_flag.loc[mask, col] = True

# 3. Check result
remaining_final = data_imputed[numeric_cols].isna().sum()
print("Remaining missing values after imputation:")
print(remaining_final[remaining_final > 0])

# 4. Count how many values were imputed per column
print("\nNumber of imputed values per column:")
print(imputed_flag.sum()[imputed_flag.sum() > 0])


Remaining missing values after imputation:
Debt service (PPG and IMF only, % of exports of goods, services and primary income)                 63
Educational attainment, Doctoral or equivalent, population 25+, female (%) (cumulative)             42
Educational attainment, Doctoral or equivalent, population 25+, male (%) (cumulative)               42
Educational attainment, Doctoral or equivalent, population 25+, total (%) (cumulative)              42
Educational attainment, at least Master's or equivalent, population 25+, female (%) (cumulative)    42
Educational attainment, at least Master's or equivalent, population 25+, male (%) (cumulative)      42
Educational attainment, at least Master's or equivalent, population 25+, total (%) (cumulative)     42
External debt stocks (% of GNI)                                                                     63
School enrollment, primary, female (% net)                                                          63
School enrollment, primary, ma

In [18]:
def compress_years(years):
    years = sorted(set(int(year) for year in years))
    if not years:
        return ""

    ranges = []
    start = prev = years[0]

    for year in years[1:]:
        if year == prev + 1:
            prev = year
        else:
            ranges.append(f"{start}-{prev}" if start != prev else str(start))
            start = prev = year

    ranges.append(f"{start}-{prev}" if start != prev else str(start))
    return ", ".join(ranges)


def describe_country_years(rows):
    if rows.empty:
        return "None"

    parts = []
    for country, group in rows.groupby("Country Name", sort=True):
        parts.append(f"{country}: {compress_years(group['Year'].tolist())}")
    return "; ".join(parts)

OBJ1_COLS = [
    "School enrollment, primary (% net)",
    "Children out of school (% of primary school age)",
    "School enrollment, secondary (% net)",
    "Adolescents out of school (% of lower secondary school age)",
]

OBJ2_COLS = [
    "Government expenditure on education, total (% of GDP)",
    "Government expenditure on education, total (% of government expenditure)",
    "School enrollment, primary (% net)",
    "Children out of school (% of primary school age)",
    "School enrollment, secondary (% net)",
    "Adolescents out of school (% of lower secondary school age)",
]

OBJ3_COLS = [
    "School enrollment, primary, female (% net)",
    "School enrollment, primary, male (% net)",
    "School enrollment, secondary, female (% net)",
    "School enrollment, secondary, male (% net)",
    "School enrollment, tertiary, female (% gross)",
    "School enrollment, tertiary, male (% gross)",
    "School enrollment, primary (gross), gender parity index (GPI)",
    "School enrollment, secondary (gross), gender parity index (GPI)",
    "School enrollment, tertiary (gross), gender parity index (GPI)",
]

OBJ4_COLS = [
    "Literacy rate, youth total (% of people ages 15-24)",
    "Literacy rate, youth (ages 15-24), gender parity index (GPI)",
    "School enrollment, secondary (% net)",
    "School enrollment, tertiary (% gross)",
]

OBJ5_COLS = [
    "Educational attainment, at least completed primary, population 25+ years, total (%) (cumulative)",
    "Educational attainment, at least completed upper secondary, population 25+, total (%) (cumulative)",
    "Educational attainment, at least Bachelor's or equivalent, population 25+, total (%) (cumulative)",
    "Educational attainment, at least Bachelor's or equivalent, population 25+, female (%) (cumulative)",
    "Educational attainment, at least Bachelor's or equivalent, population 25+, male (%) (cumulative)",
    "Educational attainment, at least Master's or equivalent, population 25+, total (%) (cumulative)",
    "Educational attainment, at least Master's or equivalent, population 25+, female (%) (cumulative)",
    "Educational attainment, at least Master's or equivalent, population 25+, male (%) (cumulative)",
    "Educational attainment, Doctoral or equivalent, population 25+, total (%) (cumulative)",
    "Educational attainment, Doctoral or equivalent, population 25+, female (%) (cumulative)",
    "Educational attainment, Doctoral or equivalent, population 25+, male (%) (cumulative)",
]

OBJ6_COLS = [
    "Life expectancy at birth, total (years)"]

OBJ7_COLS = [
    "Life expectancy at birth, total (years)",
    "GDP (current US$)",
    "GDP growth (annual %)",
    "Current health expenditure per capita, PPP (current international $)", 
    "Current health expenditure (% of GDP)",
    "Domestic general government health expenditure (% of general government expenditure)",
]

OBJ8_COLS = [
    "Mortality caused by road traffic injury (per 100,000 population)",
    "Total alcohol consumption per capita (liters of pure alcohol, projected estimates, 15+ years of age)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70 (%)"
]

OBJ9_COLS = [
    "Mortality rate, under-5 (per 1,000 live births)",
    "Mortality rate, infant (per 1,000 live births)",
    "Immunization, HepB3 (% of one-year-old children)",
    "Current health expenditure per capita, PPP (current international $)"
]

OBJ10_COLS = [
    "Life expectancy at birth, male (years)", 
    "Life expectancy at birth, female (years)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70, male (%)",
    "Mortality from CVD, cancer, diabetes or CRD between exact ages 30 and 70, female (%)",
    "Total alcohol consumption per capita, male (liters of pure alcohol, projected estimates, male 15+ years of age)",
    "Total alcohol consumption per capita, female (liters of pure alcohol, projected estimates, female 15+ years of age)"
]

OBJ11_COLS = [
    "GDP per capita (constant 2015 US$)",
    "GDP per capita growth (annual %)"
]
  
OBJ12_COLS = [  
    "Trade (% of GDP)",
    "Exports of goods and services (% of GDP)",
    "Imports of goods and services (% of GDP)"
]

OBJ13_COLS = [
    "Foreign direct investment, net inflows (% of GDP)",
    "GDP per capita growth (annual %)",
    "Exports of goods and services (% of GDP)"
]
OBJ14_COLS = [
    "External debt stocks (% of GNI)",
    "Debt service (% of exports)",
    "Total reserves (% of total external debt)",
    "Total reserves in months of imports"
]
OBJ15_COLS = [
    "Gross savings (% of GDP)",
    "Foreign direct investment, net inflows (% of GDP)",
    "Current account balance (% of GDP)"
]

OBJECTIVE_VARS = {
    "Objective 1 - Enrollment and Out-of-School": OBJ1_COLS,
    "Objective 2 - Education Spending and Access": OBJ2_COLS,
    "Objective 3 - Gender Parity in Enrollment": OBJ3_COLS,
    "Objective 4 - Youth Literacy and Higher-Level Participation": OBJ4_COLS,
    "Objective 5 - Educational Attainment": OBJ5_COLS,
    "Objective 6 - Health and Economic Indicators": OBJ6_COLS,
    "Objective 7 - Health and Economic Indicators": OBJ7_COLS,
    "Objective 8 - Health Risk Factors and Outcomes": OBJ8_COLS,
    "Objective 9 - Health Indicators": OBJ9_COLS,
    "Objective 10 - Gender-Specific Health Indicators": OBJ10_COLS,
    "Objective 11 - Economic Indicators": OBJ11_COLS,
    "Objective 12 - Trade and Economic Integration": OBJ12_COLS,
    "Objective 13 - Foreign Investment and Economic Growth": OBJ13_COLS,
    "Objective 14 - Debt Sustainability": OBJ14_COLS,
    "Objective 15 - Domestic Savings and Economic Vulnerability": OBJ15_COLS
}


analysis_ready_by_objective = {}
objective_cautions = {}
objective_missing_details = {}
objective_summary_rows = []

for objective_name, cols in OBJECTIVE_VARS.items():
    cols = [col for col in cols if col in data_imputed.columns]

    if not cols:
        analysis_ready_by_objective[objective_name] = data_imputed[["Country Name", "Year"]].copy()
        objective_missing_details[objective_name] = {}
        objective_cautions[objective_name] = (
            f"Caution for {objective_name}: no valid indicators were found in the dataset."
        )
        objective_summary_rows.append(
            {
                "Objective": objective_name,
                "Indicators used": 0,
                "Countries covered": data_imputed["Country Name"].nunique(),
                "Indicators imputed": 0,
                "Indicators still missing": 0,
            }
        )
        continue

    objective_df = data_imputed[["Country Name", "Year"] + cols].copy()
    analysis_ready_by_objective[objective_name] = objective_df

    imputed_cols = [col for col in cols if imputed_flag[col].any()]
    remaining_cols = [col for col in cols if objective_df[col].isna().any()]

    objective_missing_details[objective_name] = {
        col: objective_df.loc[objective_df[col].isna(), ["Country Name", "Year"]].copy()
        for col in remaining_cols
    }

    caution_lines = [f"Caution for {objective_name}:"]

    caution_lines = [f"Caution for {objective_name}:"]

    if imputed_cols:
        caution_lines.append("This objective uses imputed values for single-country full-series gaps:")
        for col in imputed_cols:
            rows = objective_df.loc[imputed_flag.loc[objective_df.index, col], ["Country Name", "Year"]]
            caution_lines.append(f"- {col}: {describe_country_years(rows)}")
    else:
        caution_lines.append("No imputation was applied to indicators in this objective.")

    if remaining_cols:
        caution_lines.append(
            "The following indicators still contain missing values and should be interpreted carefully "
            "when a related country appears in the analysis:"
        )
        for col in remaining_cols:
            rows = objective_missing_details[objective_name][col]
            caution_lines.append(f"- {col}: {describe_country_years(rows)}")
    else:
        caution_lines.append("No remaining missing values in this objective after safe imputation.")

    objective_cautions[objective_name] = "\n".join(caution_lines)


    objective_summary_rows.append(
        {
            "Objective": objective_name,
            "Indicators used": len(cols),
            "Countries covered": objective_df["Country Name"].nunique(),
            "Indicators imputed": len(imputed_cols),
            "Indicators still missing": len(remaining_cols),
        }
    )

objective_summary = pd.DataFrame(objective_summary_rows)


In [19]:
# Save the cleaned and merged dataset to a new CSV file
data_imputed.to_csv("SoutheastAsianData.csv", index=False)
print(f"Cleaned and merged dataset saved to SoutheastAsianData.csv")

Cleaned and merged dataset saved to SoutheastAsianData.csv


In [20]:
print("Objective summary:")
print(objective_summary.to_string(index=False))

for objective_name, caution_text in objective_cautions.items():
    print("\n" + "=" * 100)
    print(caution_text)


Objective summary:
                                                  Objective  Indicators used  Countries covered  Indicators imputed  Indicators still missing
                 Objective 1 - Enrollment and Out-of-School                4                 11                   1                         0
                Objective 2 - Education Spending and Access                6                 11                   1                         0
                  Objective 3 - Gender Parity in Enrollment                9                 11                   3                         2
Objective 4 - Youth Literacy and Higher-Level Participation                4                 11                   1                         0
                       Objective 5 - Educational Attainment               11                 11                   0                         6
               Objective 6 - Health and Economic Indicators                1                 11                   0              